---
## Decorators

A decorator is a function that **wraps another function** to extend or modify its behaviour — without changing the original function's code.

**Syntax:** `@decorator_name` placed directly above a `def` or `async def`.

In FastMCP, every tool, resource, and prompt is registered using a decorator:

```python
@mcp.tool()          # register as a callable tool
@mcp.resource(uri)   # register as readable data
@mcp.prompt()        # register as a prompt template
```

---
## JSON

JSON (JavaScript Object Notation) is the universal format for sending structured data between programs — including between Claude and your MCP server.

Every tool call, every result, every error travels as JSON over the wire. FastMCP handles the serialisation and deserialisation for you — your tool just returns a Python value.

---
## 0. `*args` and `**kwargs` — Before Decorators Make Sense



| Syntax | Collects | Type inside function |
|---|---|---|
| `*args` | extra **positional** arguments | `tuple` |
| `**kwargs` | extra **keyword** arguments | `dict` |

In [2]:
# *args — any number of positional arguments, collected into a tuple

def add(*args):
    print(f"args = {args}")   # always a tuple
    return sum(args)

print(add(1, 2))        # args = (1, 2)
print(add(1, 2, 3, 4))  # args = (1, 2, 3, 4)
print(add())            # args = ()  — zero is fine

# Normal params can come before *args
def greet(greeting, *names):
    for name in names:
        print(f"{greeting}, {name}!")

greet("Hello", "Alice", "Bob", "Carol")

args = (1, 2)
3
args = (1, 2, 3, 4)
10
args = ()
0
Hello, Alice!
Hello, Bob!
Hello, Carol!


In [3]:
# **kwargs — any number of keyword arguments, collected into a dict

def display(**kwargs):
    print(f"kwargs = {kwargs}")   # always a dict
    for key, value in kwargs.items():
        print(f"  {key}: {value}")

display(city="London", temp=65, unit="F")
display(name="Alice")
display()              # empty dict — also fine

kwargs = {'city': 'London', 'temp': 65, 'unit': 'F'}
  city: London
  temp: 65
  unit: F
kwargs = {'name': 'Alice'}
  name: Alice
kwargs = {}


In [4]:
# Both together — the universal wrapper signature used in every decorator
# Order rule: normal params → *args → **kwargs

def wrapper(*args, **kwargs):
    print(f"positional: {args}")
    print(f"keyword:    {kwargs}")

wrapper(1, 2, 3, name="Alice", city="London")



positional: (1, 2, 3)
keyword:    {'name': 'Alice', 'city': 'London'}


---
## 1. Decorators

A decorator is a function that **wraps another function** to extend or modify its behaviour — without changing the original function's code.

**Syntax:** `@decorator_name` placed directly above a `def` or `async def`.

In [5]:
# A decorator is just a function that takes a function and returns a function

def shout(func):
    """Wraps func so its return value is uppercased."""
    def wrapper(*args, **kwargs):
        result = func(*args, **kwargs)
        return result.upper()
    return wrapper

# Applying the decorator with @ syntax
@shout
def greet(name: str) -> str:
    return f"hello, {name}"

# Equivalent to: greet = shout(greet)
print(greet("alice"))   # "HELLO, ALICE"

HELLO, ALICE


In [6]:
# Without @ syntax — same thing written manually
def greet_plain(name: str) -> str:
    return f"hello, {name}"

greet_plain = shout(greet_plain)   # decorator applied manually
print(greet_plain("bob"))          # "HELLO, BOB"

HELLO, BOB


In [7]:
# *args and **kwargs are why a wrapper can wrap any function —
# it forwards everything unchanged, regardless of the original signature.

def my_decorator(func):
    def inner(*args, **kwargs):      # accept any signature
        print("before")
        result = func(*args, **kwargs)   # forward everything unchanged
        print("after")
        return result
    return inner

@my_decorator
def say(greeting, name="world"):
    print(f"{greeting}, {name}!")

say("Hello", name="Alice")

before
Hello, Alice!
after


### Decorators with arguments

Some decorators are **called with parentheses** — `@decorator(arg)`. This means the decorator is actually a factory that returns the real decorator.

In [5]:
# A decorator factory — takes config, returns a decorator
def repeat(times: int):
    def decorator(func):
        def wrapper(*args, **kwargs):
            for _ in range(times):
                result = func(*args, **kwargs)
            return result
        return wrapper
    return decorator

@repeat(times=3)
def say(msg: str):
    print(msg)

say("hello")   # prints "hello" three times

hello
hello
hello


### `@mcp.resource()` and `@mcp.prompt()` — same pattern

```python
# Resources expose read-only data (like a GET endpoint)
@mcp.resource("file://{path}")
def read_file(path: str) -> str:
    with open(path) as f:
        return f.read()

# Prompts expose reusable prompt templates
@mcp.prompt()
def summarise_prompt(text: str) -> str:
    return f"Summarise the following:

{text}"
```


---
## 2. JSON

JSON (JavaScript Object Notation) is the universal format for sending structured data between programs — including between Claude and your MCP server.

Every tool call, every result, every error travels as JSON over the wire.

In [6]:
import json

# Python dict -> JSON string (serialisation)
data = {
    "city": "London",
    "temperature": 65.3,
    "sunny": False,
    "tags": ["cloudy", "mild"],
}

json_string = json.dumps(data)
print(type(json_string))   # <class str>
print(json_string)

<class 'str'>
{"city": "London", "temperature": 65.3, "sunny": false, "tags": ["cloudy", "mild"]}


In [7]:
# JSON string -> Python dict (deserialisation)
parsed = json.loads(json_string)
print(type(parsed))          # <class dict>
print(parsed["city"])        # "London"
print(parsed["temperature"]) # 65.3

<class 'dict'>
London
65.3


In [8]:
# Pretty-printing for readability
print(json.dumps(data, indent=2))

{
  "city": "London",
  "temperature": 65.3,
  "sunny": false,
  "tags": [
    "cloudy",
    "mild"
  ]
}


### Python <-> JSON type mapping

| Python | JSON |
|--------|------|
| `dict` | object `{}` |
| `list` | array `[]` |
| `str` | string `""` |
| `int` / `float` | number |
| `True` / `False` | `true` / `false` |
| `None` | `null` |

In [9]:
# Common gotchas
import json

# None -> null
print(json.dumps({"value": None}))       # {"value": null}

# True/False -> true/false (lowercase in JSON)
print(json.dumps({"flag": True}))         # {"flag": true}

# Tuples become arrays
print(json.dumps({"point": (1.0, 2.0)}))  # {"point": [1.0, 2.0]}

# Sets are NOT serialisable
try:
    json.dumps({"items": {1, 2, 3}})
except TypeError as e:
    print(f"Error: {e}")   # sets have no JSON equivalent

{"value": null}
{"flag": true}
{"point": [1.0, 2.0]}
Error: Object of type set is not JSON serializable


### What a JSON Schema is

A JSON Schema is a JSON document that **describes the shape** of another JSON document — what fields are required, what types they must be, what values are allowed.

This is how FastMCP tells Claude what inputs your tool accepts.

In [10]:
import json

# This is a JSON Schema for a weather tool input
weather_tool_schema = {
    "type": "object",
    "properties": {
        "city": {
            "type": "string",
            "description": "The city to get weather for"
        },
        "units": {
            "type": "string",
            "description": "celsius or fahrenheit",
            "enum": ["celsius", "fahrenheit"],
            "default": "celsius"
        },
        "days": {
            "type": "integer",
            "description": "Number of forecast days",
            "minimum": 1,
            "maximum": 7
        }
    },
    "required": ["city"]
}

print(json.dumps(weather_tool_schema, indent=2))

{
  "type": "object",
  "properties": {
    "city": {
      "type": "string",
      "description": "The city to get weather for"
    },
    "units": {
      "type": "string",
      "description": "celsius or fahrenheit",
      "enum": [
        "celsius",
        "fahrenheit"
      ],
      "default": "celsius"
    },
    "days": {
      "type": "integer",
      "description": "Number of forecast days",
      "minimum": 1,
      "maximum": 7
    }
  },
  "required": [
    "city"
  ]
}


### How FastMCP generates schemas from type hints

You write Python type hints. FastMCP converts them to a JSON Schema. Claude reads the JSON Schema to know how to call the tool.

```
Your Python code          FastMCP             Claude
─────────────────         ───────────         ──────────────────────
city: str          ──>    "type": "string"  ──>  calls tool with a string
days: int          ──>    "type": "integer" ──>  calls tool with an integer
Optional[str]      ──>    not in required   ──>  knows it can be omitted
Annotated[X, msg]  ──>    "description"     ──>  knows what the field means
```

> `Optional` and `Annotated` are type hints covered in `03_type_hints.ipynb`.

In [14]:
# Simulating what FastMCP does when it sees your tool signature
import inspect
import typing

def build_schema(func) -> dict:
    hints = typing.get_type_hints(func, include_extras=True)
    schema = {"type": "object", "properties": {}, "required": []}
    sig = inspect.signature(func)

    type_map = {str: "string", int: "integer", float: "number", bool: "boolean"}

    for name, param in sig.parameters.items():
        hint = hints.get(name, str)
        description = ""

        # unwrap Annotated
        if typing.get_origin(hint) is typing.Annotated:
            args = typing.get_args(hint)
            hint, description = args[0], args[1] if len(args) > 1 else ""

        # unwrap Optional
        is_optional = False
        if typing.get_origin(hint) is typing.Union:
            inner = [a for a in typing.get_args(hint) if a is not type(None)]
            hint = inner[0] if inner else str
            is_optional = True

        json_type = type_map.get(hint, "string")
        prop = {"type": json_type}
        if description:
            prop["description"] = description
        schema["properties"][name] = prop

        if param.default is inspect.Parameter.empty and not is_optional:
            schema["required"].append(name)

    return schema

# Test it on a real function
# (Annotated and Optional are type hints — covered in full in 03_type_hints.ipynb)
from typing import Annotated, Optional

def search(
    query: Annotated[str, "The search query"],
    max_results: Annotated[int, "Max results to return"] = 10,
    language: Annotated[Optional[str], "ISO language code"] = None,
) -> list:
    pass

print(json.dumps(build_schema(search), indent=2))

{
  "type": "object",
  "properties": {
    "query": {
      "type": "string",
      "description": "The search query"
    },
    "max_results": {
      "type": "integer",
      "description": "Max results to return"
    },
    "language": {
      "type": "string",
      "description": "ISO language code"
    }
  },
  "required": [
    "query"
  ]
}


### The MCP wire protocol — JSON all the way down

Every message between Claude and your MCP server is a JSON object following the JSON-RPC 2.0 format.

**Claude calling your tool:**
```json
{
  "jsonrpc": "2.0",
  "id": 1,
  "method": "tools/call",
  "params": {
    "name": "get_weather",
    "arguments": {
      "city": "London",
      "units": "celsius"
    }
  }
}
```

**Your server responding:**
```json
{
  "jsonrpc": "2.0",
  "id": 1,
  "result": {
    "content": [
      { "type": "text", "text": "London: 65F, cloudy" }
    ]
  }
}
```

FastMCP handles all of this serialisation and deserialisation for you. Your tool just returns a Python value — FastMCP converts it to the correct JSON response automatically.

---
## 3. Both Together — Decorators and JSON

The code below shows decorators and JSON working together in the shape of a real FastMCP server.

In [11]:
import json

# ── Simulate FastMCP ─────────────────────────────────────────────
class FakeFastMCP:
    def __init__(self, name: str):
        self.name = name
        self._tools = {}

    def tool(self):
        """Decorator factory — registers the function as a tool."""
        def decorator(func):
            self._tools[func.__name__] = func
            return func
        return decorator

    def call_tool(self, name: str, arguments: dict) -> str:
        """Deserialise arguments from JSON, call tool, serialise result."""
        func = self._tools[name]
        result = func(**arguments)
        return json.dumps({"content": [{"type": "text", "text": str(result)}]})

mcp = FakeFastMCP("weather_server")

# ── Tool — registered via decorator ──────────────────────────────
@mcp.tool()
def get_weather(city: str, units: str = "celsius") -> str:
    data = {"city": city, "temp": 65, "units": units, "condition": "cloudy"}
    return json.dumps(data)

# ── Decorators and JSON together ─────────────────────────────────
# Simulate Claude calling the tool — arrives as JSON
raw_request = json.loads('{"name": "get_weather", "arguments": {"city": "London"}}')

# Call tool and get JSON response
response = mcp.call_tool(raw_request["name"], raw_request["arguments"])

print(f"Request : {json.dumps(raw_request)}")
print(f"Response: {response}")

Request : {"name": "get_weather", "arguments": {"city": "London"}}
Response: {"content": [{"type": "text", "text": "{\"city\": \"London\", \"temp\": 65, \"units\": \"celsius\", \"condition\": \"cloudy\"}"}]}
